In [1]:
import os
import pandas as df

# Load the dataset
import kagglehub
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion")

# List the file in the downloaded path to find the exact csv name
print("Files in dataset: ", os.listdir(path))

Using Colab cache for faster access to the 'conll003-englishversion' dataset.
Files in dataset:  ['valid.txt', 'metadata', 'test.txt', 'train.txt']


In [2]:
import pandas as pd

def parse_conll(file_path):
    sentences = []
    current_tokens = []

    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            # Empty lines or document markers indicate the end of a sentence
            if not line or line.startswith("-DOCSTART-"):
                if current_tokens:
                    sentences.append(" ".join(current_tokens))
                    current_tokens = []
            else:
                # CoNLL format: Word POS-Tag Chunk-Tag NER-Tag
                parts = line.split()
                if len(parts) > 0:
                    current_tokens.append(parts[0])

    return pd.DataFrame(sentences, columns=['text'])

# Load the training data
df_train = parse_conll(os.path.join(path, "train.txt"))

print(f"Total Sentences Loaded: {len(df_train)}")
print("Sample Sentence:", df_train['text'].iloc[0])

Total Sentences Loaded: 14041
Sample Sentence: EU rejects German call to boycott British lamb .


In [3]:
import spacy
from spacy.pipeline import EntityRuler

# Load the small English model
nlp = spacy.load("en_core_web_sm")

# --- Model-Based NER Approach ---

def extract_entities_model(text):
    doc = nlp(text)
    # Extract entities using the pre-trained statistical model
    entities = [(ent.text, ent.label_) for ent in doc.ents
                if ent.label_ in ['PERSON', 'GPE', 'ORG']]
    return entities

# Applying Model-Based extraction to the sample
df_sample = df_train.head(10).copy()
df_sample['entities_model'] = df_sample['text'].apply(extract_entities_model)

# Printing Model-Based results
print("--- Model-Based Results ---")
for index, row in df_sample.iterrows():
    print(f"Text: {row['text']}")
    print(f"Model Extracted: {row['entities_model']}\n")

# --- Rule-Based NER Approach ---

# Add EntityRuler to the existing pipeline to introduce custom rules
# We place it before the statistical 'ner' component so rules take priority
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Define specific patterns for exact matching (e.g., custom acronyms or specific news sources)
patterns = [
    {"label": "ORG", "pattern": "Reuters"},
    {"label": "ORG", "pattern": "U.N."},
    {"label": "GPE", "pattern": "Palestine"},
    {"label": "PERSON", "pattern": "AL-HAYAT"}
]
ruler.add_patterns(patterns)

def extract_entities_hybrid(text):
    doc = nlp(text)
    # This now extracts entities using the combined logic of rules and the model
    entities = [(ent.text, ent.label_) for ent in doc.ents
                if ent.label_ in ['PERSON', 'GPE', 'ORG']]
    return entities

# Applying the Hybrid (Rule + Model) approach to the sample
df_sample['entities_hybrid'] = df_sample['text'].apply(extract_entities_hybrid)

# Printing Hybrid results to see the difference rules made
print("--- Hybrid (Rule + Model) Results ---")
for index, row in df_sample.iterrows():
    print(f"Text: {row['text']}")
    print(f"Hybrid Extracted: {row['entities_hybrid']}\n")

--- Model-Based Results ---
Text: EU rejects German call to boycott British lamb .
Model Extracted: [('EU', 'ORG')]

Text: Peter Blackburn
Model Extracted: [('Peter Blackburn', 'PERSON')]

Text: BRUSSELS 1996-08-22
Model Extracted: [('BRUSSELS', 'GPE')]

Text: The European Commission said on Thursday it disagreed with German advice to consumers to shun British lamb until scientists determine whether mad cow disease can be transmitted to sheep .
Model Extracted: [('The European Commission', 'ORG')]

Text: Germany 's representative to the European Union 's veterinary committee Werner Zwingmann said on Wednesday consumers should buy sheepmeat from countries other than Britain until the scientific advice was clearer .
Model Extracted: [('Germany', 'GPE'), ("the European Union 's", 'ORG'), ('Werner Zwingmann', 'PERSON'), ('Britain', 'GPE')]

Text: " We do n't support any such recommendation because we do n't see any grounds for it , " the Commission 's chief spokesman Nikolaus van der Pas t

In [4]:
from spacy import displacy

# Setup filters
options = {"ents": ["PERSON", "GPE", "ORG"]}

for i, text in enumerate(df_train['text'].head(15)):
    doc = nlp(text)
    print(f"Article {i+1}")

    # Check for relevant entities
    relevant_ents = [ent for ent in doc.ents if ent.label_ in options["ents"]]

    if relevant_ents:
        displacy.render(doc, style="ent", options=options, jupyter=True)
    else:
        # print if there are not entity categories
        print(text.strip())

    print("-" * 30)

Article 1


------------------------------
Article 2


------------------------------
Article 3


------------------------------
Article 4


------------------------------
Article 5


...

------------------------------
Article 11


------------------------------
Article 12
.
------------------------------
Article 13


------------------------------
Article 14


------------------------------
Article 15


------------------------------


In [5]:
# --- Rule Based Approach with large Spacy Model ---
!python -m spacy download en_core_web_lg

import spacy
from spacy.pipeline import EntityRuler

# Load the large model
nlp_lg = spacy.load("en_core_web_lg")

# Add the rule-based component
ruler = nlp_lg.add_pipe("entity_ruler", before="ner")

# Define custom patterns
patterns = [
    {"label": "ORG", "pattern": "Reuters"},
    {"label": "ORG", "pattern": "U.N."},
    {"label": "GPE", "pattern": "Palestine"},
    {"label": "PERSON", "pattern": "AL-HAYAT"}
]
ruler.add_patterns(patterns)

# Hybrid extraction function using the large model
def extract_lg_hybrid(text):
    doc = nlp_lg(text)
    return [(ent.text, ent.label_) for ent in doc.ents
            if ent.label_ in ['PERSON', 'GPE', 'ORG']]

# Apply to first 10 rows
df_sample = df_train.head(10).copy()
df_sample['entities_lg'] = df_sample['text'].apply(extract_lg_hybrid)

# Display results
for index, row in df_sample.iterrows():
    print(f"Text: {row['text']}")
    print(f"Large Hybrid Extracted: {row['entities_lg']}\n")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 750.6 kB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Text: EU rejects German call to boycott British lamb .
Large Hybrid Extracted: [('EU', 'ORG')]

Text: Peter Blackburn
Large Hybrid Extracted: [('Peter Blackburn', 'PERSON')]

Text: BRUSSELS 1996-08-22
Large Hybrid Extracted: [('BRUSSELS', 'GPE')]

Text: The European Commission said on Thursday it disagreed with German advice to consumers to shun British lamb until scientists determine whether mad cow disease can be transmitted to sheep .
Large Hybrid Extracted: [('The European Commission', 'ORG')]

Text: Germany 's representative to the European Union 's veterinary committee W

In [6]:
from spacy import displacy

# Set visualization categories
options = {"ents": ["PERSON", "GPE", "ORG"]}

# Visualize the first 10 articles from your sample
for i, text in enumerate(df_sample['text']):
    # Process text with the hybrid large model
    doc = nlp_lg(text)

    print(f"Article {i+1} (Large Hybrid)")

    # Check if target entities exist
    if any(ent.label_ in options["ents"] for ent in doc.ents):
        displacy.render(doc, style="ent", options=options, jupyter=True)
    else:
        print(text.strip())

    print("-" * 30)

Article 1 (Large Hybrid)


------------------------------
Article 2 (Large Hybrid)


------------------------------
Article 3 (Large Hybrid)


------------------------------
Article 4 (Large Hybrid)


------------------------------
Article 5 (Large Hybrid)


...

------------------------------
Article 7 (Large Hybrid)


------------------------------
Article 8 (Large Hybrid)


------------------------------
Article 9 (Large Hybrid)


------------------------------
Article 10 (Large Hybrid)


------------------------------


In [7]:
# Select target labels
labels = ['PERSON', 'GPE', 'ORG']

# Compare first 15 articles
for i, text in enumerate(df_train['text'].head(15)):
    # Get entities from Small model (nlp)
    doc_sm = nlp(text)
    sm_ents = [f"{e.text} ({e.label_})" for e in doc_sm.ents if e.label_ in labels]

    # Get entities from Large model (nlp_lg)
    doc_lg = nlp_lg(text)
    lg_ents = [f"{e.text} ({e.label_})" for e in doc_lg.ents if e.label_ in labels]

    print(f"--- Article {i+1} ---")
    print(f"SM Model: {sm_ents}")
    print(f"LG Model: {lg_ents}")
    print("\n")

--- Article 1 ---
SM Model: ['EU (ORG)']
LG Model: ['EU (ORG)']


--- Article 2 ---
SM Model: ['Peter Blackburn (PERSON)']
LG Model: ['Peter Blackburn (PERSON)']


--- Article 3 ---
SM Model: ['BRUSSELS (GPE)']
LG Model: ['BRUSSELS (GPE)']


--- Article 4 ---
SM Model: ['The European Commission (ORG)']
LG Model: ['The European Commission (ORG)']


--- Article 5 ---
SM Model: ['Germany (GPE)', "the European Union 's (ORG)", 'Werner Zwingmann (PERSON)', 'Britain (GPE)']
LG Model: ['Germany (GPE)', "the European Union 's (ORG)", 'Werner Zwingmann (PERSON)', 'Britain (GPE)']


--- Article 6 ---
SM Model: ['Commission (ORG)', 'Nikolaus van der Pas (PERSON)']
LG Model: ['Commission (ORG)', 'Nikolaus van der Pas (PERSON)']


--- Article 7 ---
SM Model: ['the European Union (ORG)']
LG Model: ['the European Union (ORG)']


--- Article 8 ---
SM Model: ['EU Farm (ORG)', 'Franz Fischler (PERSON)']
LG Model: ['EU Farm (ORG)', 'Franz Fischler (PERSON)']


--- Article 9 ---
SM Model: ['Fischler (PERS

In [8]:
# Load Validation data (to tune your rules)
df_valid = parse_conll(os.path.join(path, "valid.txt"))

# Load Test data (for the final grade)
df_test = parse_conll(os.path.join(path, "test.txt"))

print(f"Validation Sentences: {len(df_valid)}")
print(f"Test Sentences: {len(df_test)}")

Validation Sentences: 3250
Test Sentences: 3453
